# Certify the fault distance of circuits

The fault distance of a noisy circuit is the smallest number of faults that flip a logical observable without firing any detector.
A fault is one error mechanism of the circuit's detector error model.
Computing it exactly is expensive: both brute-force enumeration and `circuits.get_fault_distance` take exponential time in the worst case.
Enumeration checks every combination of faults up to the fault distance.
`circuits.get_fault_distance` instead converts the circuit into an auxiliary CSS code and runs the Brouwer-Zimmermann code-distance algorithm, which stops as soon as it can prove that no lighter undetected fault exists.
The cost still grows quickly with the fault distance and the number of fault mechanisms, so large circuits can be out of reach.
See `help(circuits.get_fault_distance)` for the construction.

The examples below compute the fault distance of memory experiments for a surface code and a Steane code.

## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.

In [1]:
%%capture
%pip install qldpc

In [2]:
from qldpc import circuits, codes

## Certify the fault distance of a memory experiment

Here we compute the fault distance of an X- and Z-basis memory experiment for a distance-5 surface code with five rounds of syndrome extraction.
Stim's graphlike search for undetectable errors provides a cross-check for this example because every circuit fault in this example flips at most two detectors.
Unlike Stim's graphlike search, however, `circuits.get_fault_distance` does not require graphlike faults.

In [3]:
code = codes.SurfaceCode(5)
noise_model = circuits.DepolarizingNoiseModel(1e-3)

for basis in ("X", "Z"):
    # one logical memory cycle = code.get_distance() rounds of syndrome extraction
    # you can replace logical_memory_cycle with another memory circuit
    initialization, logical_memory_cycle, readout, *_ = circuits.get_memory_experiment_parts(
        code, basis=basis, num_rounds=code.get_distance()
    )

    # keep initialization and readout noiseless, to certify only the inner QEC cycle
    noisy_circuit = initialization + noise_model.noisy_circuit(logical_memory_cycle) + readout
    fault_distance = circuits.get_fault_distance(noisy_circuit)
    graphlike_distance = len(noisy_circuit.shortest_graphlike_error())
    assert fault_distance == graphlike_distance
    print(
        f"{basis}-basis memory: fault distance {fault_distance}, "
        f"Stim graphlike distance {graphlike_distance}"
    )

X-basis memory: fault distance 5, Stim graphlike distance 5


Z-basis memory: fault distance 5, Stim graphlike distance 5


## Certify a memory experiment with non-graphlike faults

A fault is graphlike if it flips at most two detectors.
In a memory experiment for the Steane code, some faults flip three detectors.
We build the experiment with the same helpers and print the largest number of detectors that a single fault flips in the detector error model, next to the fault distance.

In [4]:
code = codes.SteaneCode()

for basis in ("X", "Z"):
    initialization, logical_memory_cycle, readout, *_ = circuits.get_memory_experiment_parts(
        code, basis=basis, num_rounds=code.get_distance()
    )
    noisy_circuit = initialization + noise_model.noisy_circuit(logical_memory_cycle) + readout

    # the number of detectors flipped by each fault of the detector error model
    detectors_per_fault = [
        sum(target.is_relative_detector_id() for target in instruction.targets_copy())
        for instruction in noisy_circuit.detector_error_model().flattened()
        if instruction.type == "error"
    ]
    fault_distance = circuits.get_fault_distance(noisy_circuit)
    print(
        f"{basis}-basis memory: a fault flips up to {max(detectors_per_fault)} detectors, "
        f"fault distance {fault_distance}, code distance {code.get_distance()}"
    )

X-basis memory: a fault flips up to 3 detectors, fault distance 2, code distance 3


Z-basis memory: a fault flips up to 3 detectors, fault distance 2, code distance 3


Some faults of this circuit are not graphlike, yet `circuits.get_fault_distance` applies without change.
It works with the detector and observable flip matrices of the detector error model, in which a fault can flip any number of detectors.